In [1]:
!pip install -q gradio matplotlib numpy

import gradio as gr
import matplotlib.pyplot as plt
import numpy as np


def simulate_foam_glider_ar(
    ar_low, ar_mid, ar_high,
    oswald_e, mass_kg, wing_area_m2,
    start_alt_m, launch_speed_ms, target_dist_m,
    speed_sigma_ms, angle_sigma_deg, n_trials
):
    rho = 1.225
    g = 9.80665
    weight_n = max(mass_kg, 0.05) * g
    s_ref = max(wing_area_m2, 0.01)
    h0 = max(start_alt_m, 0.5)
    n_runs = int(np.clip(n_trials, 50, 1000))

    ar_configs = [
        ("Low AR", float(ar_low), "#f87171"),
        ("Mid AR", float(ar_mid), "#fbbf24"),
        ("High AR", float(ar_high), "#34d399"),
    ]

    rng = np.random.default_rng(42)
    v0_samples = np.maximum(2.0, launch_speed_ms + rng.normal(0.0, max(speed_sigma_ms, 0.05), n_runs))
    ang_samples_deg = launch_angle_deg = launch_speed_ms * 0.0 + rng.normal(0.0, max(angle_sigma_deg, 0.1), n_runs)
    # Gust / wind effect on effective glide ratio (small stochastic perturbation)
    gust_factor = np.clip(1.0 + rng.normal(0.0, 0.06, n_runs), 0.72, 1.28)

    fig, (ax_dist, ax_acc) = plt.subplots(1, 2, figsize=(13.2, 4.9), dpi=130)
    fig.patch.set_facecolor("#0f172a")
    for ax in (ax_dist, ax_acc):
        ax.set_facecolor("#0f172a")
        ax.tick_params(colors="#94a3b8", labelsize=8)
        for spine in ax.spines.values():
            spine.set_color("#334155")

    summary_lines = []

    for label, ar_val, color in ar_configs:
        ar_safe = max(ar_val, 1.5)
        e_safe = np.clip(oswald_e, 0.45, 0.95)

        # Typical foam glider operating lift coefficient & parasitic drag
        cl_op = 0.58
        cd0 = 0.032 + 0.0015 * max(ar_safe - 5.0, 0.0)
        cd_ind = (cl_op ** 2) / (np.pi * ar_safe * e_safe)
        ld_nominal = cl_op / max(cd0 + cd_ind, 1e-4)

        # Initial kinetic energy contribution + potential energy glide distance
        ke_equiv_m = 0.5 * (v0_samples ** 2) / g
        pitch_bonus_m = np.tan(np.radians(ang_samples_deg)) * (h0 * 0.25)
        glide_dist_m = np.maximum(0.5, (h0 + ke_equiv_m * 0.35 + pitch_bonus_m) * ld_nominal * gust_factor)

        landing_err_m = np.abs(glide_dist_m - target_dist_m)
        mean_dist = float(np.mean(glide_dist_m))
        std_dist = float(np.std(glide_dist_m))
        mean_err = float(np.mean(landing_err_m))
        # Repeatability score (0-100%): higher when relative spread (std / mean) is low
        repeatability_pct = float(np.clip(100.0 * (1.0 - (std_dist / max(mean_dist, 1.0)) * 2.5), 0.0, 100.0))

        ax_dist.hist(
            glide_dist_m, bins=22, alpha=0.55, color=color,
            edgecolor="#0f172a", label=f"{label} (AR={ar_safe:.1f}, mean={mean_dist:.1f} m)"
        )
        ax_dist.axvline(mean_dist, color=color, linestyle="--", lw=1.6)

        trial_idx = np.arange(1, n_runs + 1)
        ax_acc.plot(trial_idx, landing_err_m, ".", color=color, alpha=0.45, markersize=3.5, label=f"{label} (err={mean_err:.1f} m)")

        summary_lines.append(
            f"- **{label} (AR = {ar_safe:.1f}):** Nominal L/D ≈ {ld_nominal:.1f}, "
            f"Mean Glide Distance ≈ {mean_dist:.1f} m (± {std_dist:.1f} m std), "
            f"Mean Landing Error ≈ {mean_err:.1f} m, Repeatability ≈ {repeatability_pct:.0f}%."
        )

    ax_dist.axvline(target_dist_m, color="#38bdf8", linestyle="-", lw=1.8, label=f"Target ({target_dist_m:.0f} m)")
    ax_dist.set_title("1. Glide Distance Distribution Across Trials", color="#f1f5f9", fontsize=10, pad=6)
    ax_dist.set_xlabel("Glide Distance (m)", color="#cbd5e1", fontsize=8.5)
    ax_dist.set_ylabel("Trial Count", color="#cbd5e1", fontsize=8.5)
    ax_dist.legend(loc="upper right", fontsize=7, facecolor="#1e293b", edgecolor="#475569", labelcolor="#f8fafc")

    ax_acc.axhline(0.0, color="#64748b", linestyle=":", lw=0.9)
    ax_acc.set_title("2. Landing Distance Error vs. Target Across Trials", color="#f1f5f9", fontsize=10, pad=6)
    ax_acc.set_xlabel("Trial Number", color="#cbd5e1", fontsize=8.5)
    ax_acc.set_ylabel("Absolute Target Error |d - d_target| (m)", color="#cbd5e1", fontsize=8.5)
    ax_acc.legend(loc="upper right", fontsize=7, facecolor="#1e293b", edgecolor="#475569", labelcolor="#f8fafc")

    fig.suptitle(
        f"Foam Glider Aspect Ratio Study  |  h0={h0:.1f} m, V0={launch_speed_ms:.1f} m/s, Target={target_dist_m:.0f} m ({n_runs} trials)",
        color="#f8fafc", fontsize=11, fontweight="bold", y=0.98
    )
    plt.tight_layout(rect=[0, 0, 1, 0.94])

    md_summary = (
        "### 🛩️ Aspect Ratio, Landing Accuracy & Repeatability Summary\n"
        + "\n".join(summary_lines)
        + "\n\n**Engineering Takeaway:**\n"
        "- Higher Aspect Ratio reduces induced drag and boosts nominal glide distance, but wider spans can amplify "
        "wind/gust or launch-angle sensitivity on lightweight foam gliders.\n"
        "- Comparing mean target error and trial-to-trial standard deviation helps balance maximum range against landing consistency."
    )

    return fig, md_summary


with gr.Blocks() as demo:
    gr.Markdown("## 🛩️ Foam Glider Wing Aspect Ratio & Repeatability Simulator")

    with gr.Row():
        with gr.Column(scale=1):
            ar_low = gr.Slider(2.5, 6.0, value=4.0, step=0.5, label="Low Aspect Ratio (AR_low)")
            ar_mid = gr.Slider(5.5, 9.5, value=7.5, step=0.5, label="Mid Aspect Ratio (AR_mid)")
            ar_high = gr.Slider(9.0, 16.0, value=12.0, step=0.5, label="High Aspect Ratio (AR_high)")
            oswald_e = gr.Slider(0.55, 0.92, value=0.78, step=0.02, label="Oswald Efficiency Factor (e)")
            mass_kg = gr.Slider(0.05, 1.50, value=0.25, step=0.05, label="Foam Glider Mass (kg)")
            wing_area_m2 = gr.Slider(0.03, 0.60, value=0.14, step=0.01, label="Wing Area, S (m²)")
            start_alt_m = gr.Slider(1.0, 25.0, value=4.0, step=0.5, label="Launch / Drop Altitude, h0 (m)")
            launch_speed_ms = gr.Slider(3.0, 18.0, value=8.0, step=0.5, label="Launch Speed, V0 (m/s)")
            target_dist_m = gr.Slider(5.0, 120.0, value=35.0, step=5.0, label="Target Landing Distance (m)")
            speed_sigma_ms = gr.Slider(0.1, 3.0, value=0.6, step=0.1, label="Launch Speed Scatter, sigma_V (m/s)")
            angle_sigma_deg = gr.Slider(0.5, 8.0, value=2.5, step=0.5, label="Launch Angle Scatter, sigma_ang (deg)")
            n_trials = gr.Slider(100, 800, value=300, step=50, label="Monte Carlo Trials")

            run_btn = gr.Button("Simulate Glider Trials", variant="primary")

        with gr.Column(scale=2):
            glider_plot = gr.Plot(label="Glide Distance & Landing Error Analysis")
            glider_md = gr.Markdown()

    inputs = [
        ar_low, ar_mid, ar_high, oswald_e, mass_kg, wing_area_m2,
        start_alt_m, launch_speed_ms, target_dist_m,
        speed_sigma_ms, angle_sigma_deg, n_trials
    ]
    run_btn.click(fn=simulate_foam_glider_ar, inputs=inputs, outputs=[glider_plot, glider_md])
    demo.load(fn=simulate_foam_glider_ar, inputs=inputs, outputs=[glider_plot, glider_md])

demo.launch(share=True, theme=gr.themes.Soft())

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://267cf7bbaacdc2bae7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
